In [0]:
flights_df = spark.table("workspace.default.flights_raw")
airlines_df = spark.table("workspace.default.airlines_raw")
airports_df = spark.table("workspace.default.airports_raw")

print("Flights:", flights_df.count())
print("Airlines:", airlines_df.count())
print("Airports:", airports_df.count())

Flights: 5819079
Airlines: 14
Airports: 322


In [0]:
from pyspark.sql.functions import col, when

# Fill null delay values with 0, otherwise calculations will break
flights_clean = flights_df.fillna({"DEPARTURE_DELAY": 0, "ARRIVAL_DELAY": 0, "CANCELLED": 0})

# Keep only the columns we actually need out of the original 31
cols_needed = ["YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "FLIGHT_NUMBER",
               "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DEPARTURE_DELAY", "ARRIVAL_DELAY", "CANCELLED"]

flights_clean = flights_clean.select(*cols_needed)

# A flight is considered "delayed" if it arrives 15+ minutes late (standard industry definition)
flights_clean = flights_clean.withColumn("IS_DELAYED", when(col("ARRIVAL_DELAY") >= 15, 1).otherwise(0))

print("Total rows after cleaning:", flights_clean.count())
flights_clean.show(5)

Total rows after cleaning: 5819079
+----+-----+---+-----------+-------+-------------+--------------+-------------------+---------------+-------------+---------+----------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|DEPARTURE_DELAY|ARRIVAL_DELAY|CANCELLED|IS_DELAYED|
+----+-----+---+-----------+-------+-------------+--------------+-------------------+---------------+-------------+---------+----------+
|2015|    1|  1|          4|     AS|           98|           ANC|                SEA|            -11|          -22|        0|         0|
|2015|    1|  1|          4|     AA|         2336|           LAX|                PBI|             -8|           -9|        0|         0|
|2015|    1|  1|          4|     US|          840|           SFO|                CLT|             -2|            5|        0|         0|
|2015|    1|  1|          4|     AA|          258|           LAX|                MIA|             -5|           -9|        0|         0|
|2015|

In [0]:
# Save the cleaned data as a Delta table so it persists beyond this notebook
flights_clean.write.format("delta").mode("overwrite").saveAsTable("workspace.default.flights_silver")

print("Silver table created: flights_silver")

In [0]:
# Register the silver table as a temporary SQL view so we can query it with SQL
flights_clean.createOrReplaceTempView("flights_temp")


In [0]:
airline_delay_summary = spark.sql("""
    SELECT AIRLINE,
           COUNT(*) AS total_flights,
           SUM(IS_DELAYED) AS total_delayed,
           ROUND(AVG(ARRIVAL_DELAY), 2) AS avg_arrival_delay
    FROM flights_temp
    GROUP BY AIRLINE
    ORDER BY avg_arrival_delay DESC
""")

airline_delay_summary.show()

+-------+-------------+-------------+-----------------+
|AIRLINE|total_flights|total_delayed|avg_arrival_delay|
+-------+-------------+-------------+-----------------+
|     NK|       117379|        34221|             14.2|
|     F9|        90836|        23570|             12.4|
|     B6|       267048|        59175|             6.55|
|     EV|       571977|       109184|             6.39|
|     MQ|       294632|        60547|             6.11|
|     OO|       588353|       107795|             5.73|
|     UA|       515723|       104722|             5.35|
|     VX|        61903|        11778|             4.69|
|     WN|      1261855|       236626|             4.31|
|     US|       198715|        36549|             3.62|
|     AA|       725984|       130279|             3.39|
|     HA|        76272|         8618|             2.02|
|     DL|       875881|       118023|             0.19|
|     AS|       172521|        22352|            -0.97|
+-------+-------------+-------------+-----------

In [0]:
# Find airports where the average delay is worse than the overall average delay
# This uses a subquery — the inner query calculates the overall average first
airport_delay_analysis = spark.sql("""
    SELECT ORIGIN_AIRPORT,
           COUNT(*) AS total_flights,
           ROUND(AVG(DEPARTURE_DELAY), 2) AS avg_departure_delay
    FROM flights_temp
    GROUP BY ORIGIN_AIRPORT
    HAVING AVG(DEPARTURE_DELAY) > (
        SELECT AVG(DEPARTURE_DELAY) FROM flights_temp
    )
    ORDER BY avg_departure_delay DESC
    LIMIT 15
""")

airport_delay_analysis.show()


+--------------+-------------+-------------------+
|ORIGIN_AIRPORT|total_flights|avg_departure_delay|
+--------------+-------------+-------------------+
|         14222|            9|              89.11|
|           ILG|          100|              28.51|
|           MVY|          205|              25.91|
|         13964|           36|              25.64|
|           HYA|           83|               22.9|
|         10154|           28|              22.86|
|         10581|           27|              20.11|
|           STC|           83|              17.57|
|         10165|            9|              17.56|
|         14025|           13|              17.54|
|           OTH|          275|              17.13|
|           GST|           77|              16.95|
|           GUM|          334|              16.55|
|           ASE|         3562|              16.24|
|           ACK|          492|              16.19|
+--------------+-------------+-------------------+



In [0]:
# Save both insights as Gold tables so they persist and can be used in a dashboard later
airline_delay_summary.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_airline_delays")
airport_delay_analysis.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_airport_delays")

print("Gold tables created: gold_airline_delays, gold_airport_delays")

Gold tables created: gold_airline_delays, gold_airport_delays


In [0]:
monthly_delay_trend = spark.sql("""
    SELECT MONTH,
           COUNT(*) AS total_flights,
           SUM(IS_DELAYED) AS total_delayed,
           ROUND(SUM(IS_DELAYED) * 100.0 / COUNT(*), 2) AS delay_percentage
    FROM flights_temp
    GROUP BY MONTH
    ORDER BY MONTH
""")

monthly_delay_trend.show(12)

# Save this as a Gold table too
monthly_delay_trend.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_monthly_delays")
print("Gold table created: gold_monthly_delays")

+-----+-------------+-------------+----------------+
|MONTH|total_flights|total_delayed|delay_percentage|
+-----+-------------+-------------+----------------+
|    1|       469968|        95951|           20.42|
|    2|       429191|        95179|           22.18|
|    3|       504312|        95452|           18.93|
|    4|       485151|        82247|           16.95|
|    5|       496993|        89645|           18.04|
|    6|       503897|       115742|           22.97|
|    7|       520718|       107627|           20.67|
|    8|       510536|        94113|           18.43|
|    9|       464946|        60061|           12.92|
|   10|       486165|        60079|           12.36|
|   11|       467972|        70571|           15.08|
|   12|       479230|        96772|           20.19|
+-----+-------------+-------------+----------------+

Gold table created: gold_monthly_delays


In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Convert airline (text) into a numeric code, since ML models only understand numbers
airline_indexer = StringIndexer(inputCol="AIRLINE", outputCol="AIRLINE_INDEX")

# Combine all the features we want to use into a single vector column
assembler = VectorAssembler(
    inputCols=["MONTH", "DAY_OF_WEEK", "AIRLINE_INDEX", "DEPARTURE_DELAY"],
    outputCol="features"
)

# The actual model
lr = LogisticRegression(featuresCol="features", labelCol="IS_DELAYED")

# Chain all steps together into one pipeline
pipeline = Pipeline(stages=[airline_indexer, assembler, lr])

# Split data into training (80%) and testing (20%)
train_data, test_data = flights_clean.randomSplit([0.8, 0.2], seed=42)

# Train the model
model = pipeline.fit(train_data)

print("Model training done")

Model training done


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Run the trained model on the test data (data it hasn't seen before)
predictions = model.transform(test_data)

# Check how many predictions were correct
evaluator = MulticlassClassificationEvaluator(
    labelCol="IS_DELAYED",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Look at a few actual predictions
predictions.select("MONTH", "AIRLINE", "DEPARTURE_DELAY", "IS_DELAYED", "prediction").show(10)

Model Accuracy: 93.00%
+-----+-------+---------------+----------+----------+
|MONTH|AIRLINE|DEPARTURE_DELAY|IS_DELAYED|prediction|
+-----+-------+---------------+----------+----------+
|    1|     AA|            -11|         0|       0.0|
|    1|     AA|             -6|         0|       0.0|
|    1|     AA|             31|         1|       1.0|
|    1|     AA|             -6|         0|       0.0|
|    1|     AA|             39|         1|       1.0|
|    1|     AA|             -4|         0|       0.0|
|    1|     AA|              0|         0|       0.0|
|    1|     AA|             -4|         0|       0.0|
|    1|     AA|             -1|         1|       0.0|
|    1|     AA|              0|         0|       0.0|
+-----+-------+---------------+----------+----------+
only showing top 10 rows


In [0]:
# Quick look at all our gold tables together before visualizing
display(spark.table("workspace.default.gold_airline_delays"))

AIRLINE,total_flights,total_delayed,avg_arrival_delay
NK,117379,34221,14.2
F9,90836,23570,12.4
B6,267048,59175,6.55
EV,571977,109184,6.39
MQ,294632,60547,6.11
OO,588353,107795,5.73
UA,515723,104722,5.35
VX,61903,11778,4.69
WN,1261855,236626,4.31
US,198715,36549,3.62


Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table("workspace.default.gold_monthly_delays"))

MONTH,total_flights,total_delayed,delay_percentage
1,469968,95951,20.42
2,429191,95179,22.18
3,504312,95452,18.93
4,485151,82247,16.95
5,496993,89645,18.04
6,503897,115742,22.97
7,520718,107627,20.67
8,510536,94113,18.43
9,464946,60061,12.92
10,486165,60079,12.36


Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table("workspace.default.gold_airport_delays"))


ORIGIN_AIRPORT,total_flights,avg_departure_delay
14222,9,89.11
ILG,100,28.51
MVY,205,25.91
13964,36,25.64
HYA,83,22.9
10154,28,22.86
10581,27,20.11
STC,83,17.57
10165,9,17.56
14025,13,17.54


Databricks visualization. Run in Databricks to view.